In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn


## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


## Chargement des donnees


In [ ]:
import os

# Support execution from repository root or notebooks/ directory
data_path = "diamonds.csv" if os.path.exists("diamonds.csv") else "../diamonds.csv"

try:
    df = pd.read_csv(data_path)
except Exception:
    df = sns.load_dataset("diamonds")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df.head()


In [ ]:
df.shape


## 1. Exploration des donnees (EDA)


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
# Verification des valeurs manquantes et doublons
print("Valeurs manquantes :\n", df.isnull().sum())
print("Nombre de doublons :", df.duplicated().sum())


In [ ]:
# Distribution du prix
plt.figure(figsize=(7, 4))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribution du prix")
plt.show()


In [ ]:
# Relations avec le prix
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="carat", y="price", alpha=0.3)
plt.title("Prix vs Carat")
plt.show()


In [ ]:
# Matrice de correlation
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation des variables numeriques")
plt.show()


## 2. Nettoyage des donnees


In [ ]:
# Suppression des doublons
df = df.drop_duplicates()

# Correction des dimensions impossibles (x, y ou z = 0)
df = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)]

print("Dimensions apres nettoyage :", df.shape)


## 3. Preparation et Split Train / Test


In [ ]:
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")


## 4. Preprocessing & ColumnTransformer


In [ ]:
num_features = ["carat", "depth", "table", "x", "y", "z"]
ord_features = ["cut", "color", "clarity"]

cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_order = ["J", "I", "H", "G", "F", "E", "D"]
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[cut_order, color_order, clarity_order]))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, ord_features)
])


## 5. Algorithme 1 : Linear Regression


In [ ]:
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Validation croisee (5 folds)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
lr_cv_scores = cross_val_score(lr_pipeline, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")

print(f"Linear Regression - RMSE CV moyen : {-lr_cv_scores.mean():.2f} (+/- {lr_cv_scores.std():.2f})")

# Fit du modele complet
lr_pipeline.fit(X_train, y_train)
